# RAG 第 17 课：Contextual Compression（上下文压缩）

前面检索完之后，我们通常一股脑把 top-k 整段塞给 LLM。
这会带来两个麻烦：

1. token 烧得很快
2. 相关信息被大量无关文字"稀释"，LLM 反而更容易走神

这一课我们在"检索完"和"生成答案"之间加一步：

**Contextual Compression = 只保留对当前问题真正有用的句子。**

做法有两种主流：
- **Extractive**：让 LLM 从原文里**挑句子**出来，不改写
- **Abstractive**：让 LLM **重写**成更短的摘要

这节课我们都做，并和"不压缩"对比。

In [ ]:
import gc
try:
    import torch
    if torch.cuda.is_available():
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()
        print('GPU cleared.')
    else: print('CUDA not available.')
except Exception as e: print(f'no torch ({e})')

In [ ]:
import math, re
from collections import Counter, defaultdict
from typing import List
import numpy as np
from datasets import load_dataset
from openai import OpenAI

client = OpenAI(base_url='http://10.0.0.63:1234/v1', api_key='lm-studio')
model_ids = [m.id for m in client.models.list().data]
preferred = ['qwen/qwen3.6-35b-a3b', 'qwen3.6-35b-a3b', 'qwen3.5-35b-a3b', 'qwen3.5-27b']
chat_model = next((m for m in preferred if m in model_ids), None)
embedding_model = next((m for m in model_ids if 'embed' in m.lower()), None)
print('chat =', chat_model, '| embed =', embedding_model)

## 1. 数据 + 基础检索

为了让压缩效果更明显，这节课我们故意用**更长的 top-k**（比如 5 个文档），让上下文看起来臃肿。

In [ ]:
raw_ds = load_dataset('squad', split='validation[:160]')
context_to_doc_id = {}; documents=[]; eval_rows=[]
for item in raw_ds:
    c = item['context'].strip()
    if c not in context_to_doc_id:
        context_to_doc_id[c] = len(documents)
        documents.append({'doc_id': len(documents), 'text': c, 'title': item['title']})
    did = context_to_doc_id[c]
    if item['answers']['text']:
        eval_rows.append({'question': item['question'].strip(), 'gold_doc_id': did,
                          'reference_answer': item['answers']['text'][0].strip(), 'title': item['title']})
print('docs=', len(documents), 'eval=', len(eval_rows))

In [ ]:
def tokenize(t): return re.findall(r'[a-zA-Z0-9]+', t.lower())

class BM25:
    def __init__(self, corpus, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.doc_lens = np.array([len(t) for t in corpus], dtype=np.float32)
        self.avgdl = float(self.doc_lens.mean()); self.N = len(corpus)
        self.postings = defaultdict(dict)
        for did, toks in enumerate(corpus):
            for tok, f in Counter(toks).items():
                self.postings[tok][did] = f
        self.idf = {t: math.log(1+(self.N-len(p)+0.5)/(len(p)+0.5)) for t, p in self.postings.items()}
    def score(self, qt):
        s = np.zeros(self.N, dtype=np.float32)
        for tok in qt:
            if tok not in self.postings: continue
            idf = self.idf[tok]
            for did, f in self.postings[tok].items():
                dl = self.doc_lens[did]
                s[did] += idf*(f*(self.k1+1))/(f + self.k1*(1-self.b+self.b*dl/self.avgdl))
        return s

doc_tokens = [tokenize(d['text']) for d in documents]
bm25 = BM25(doc_tokens)

def get_emb(texts, bs=16):
    out=[]
    for i in range(0, len(texts), bs):
        r = client.embeddings.create(model=embedding_model, input=texts[i:i+bs])
        out.extend([np.array(x.embedding, dtype=np.float32) for x in r.data])
    return np.vstack(out)

def l2n(m):
    n = np.linalg.norm(m, axis=1, keepdims=True); return m/np.clip(n, 1e-12, None)

doc_embeddings = l2n(get_emb([d['text'] for d in documents]))

def hybrid(q, top_k=5, cand=20):
    lex_s = bm25.score(tokenize(q))
    lex_idx = np.argsort(lex_s)[::-1][:cand]
    qe = l2n(get_emb([q]))[0]; den_s = doc_embeddings @ qe
    den_idx = np.argsort(den_s)[::-1][:cand]
    fused = defaultdict(float)
    for rk,i in enumerate(lex_idx, 1): fused[int(i)] += 1/(60+rk)
    for rk,i in enumerate(den_idx, 1): fused[int(i)] += 1/(60+rk)
    return [d for d,_ in sorted(fused.items(), key=lambda x:x[1], reverse=True)[:top_k]]

## 2. 压缩方式 A：Extractive（抽句子）

让 LLM 从每个文档里**只挑与问题相关的完整句子**出来，不改写、不拼接。
这种方式最保守，最不容易引入幻觉。

In [ ]:
def extractive_compress(question: str, doc_id: int) -> str:
    text = documents[doc_id]['text']
    sys = ('You extract only the sentences from the passage that help answer the question. '
           'Rules: (1) copy sentences verbatim, do not rewrite; '
           '(2) preserve original order; '
           '(3) if nothing is relevant, output the single word NONE; '
           '(4) output sentences only, no commentary.')
    usr = f'Question: {question}\n\nPassage:\n{text}\n\nRelevant sentences:'
    r = client.chat.completions.create(model=chat_model, temperature=0,
        messages=[{'role':'system','content':sys},{'role':'user','content':usr}])
    out = r.choices[0].message.content.strip()
    return '' if out.upper().startswith('NONE') else out

## 3. 压缩方式 B：Abstractive（重写摘要）

让 LLM 针对问题把每个文档**重写成一句话摘要**。
更短、信息密度更高，但有一定幻觉风险，所以 prompt 要明确要求"只基于原文"。

In [ ]:
def abstractive_compress(question: str, doc_id: int) -> str:
    text = documents[doc_id]['text']
    sys = ('You write one sentence that summarizes what the passage says about the question. '
           'Rules: (1) base the summary strictly on the passage, no outside knowledge; '
           '(2) if the passage does not address the question, output NONE; '
           '(3) output the single sentence only.')
    usr = f'Question: {question}\n\nPassage:\n{text}\n\nOne-sentence summary:'
    r = client.chat.completions.create(model=chat_model, temperature=0,
        messages=[{'role':'system','content':sys},{'role':'user','content':usr}])
    out = r.choices[0].message.content.strip()
    return '' if out.upper().startswith('NONE') else out

## 4. 把压缩结果拼回 context

In [ ]:
def build_raw_context(doc_ids):
    return '\n\n'.join([f'[Document {i}] {documents[d]["text"]}' for i,d in enumerate(doc_ids,1)])

def build_compressed_context(question, doc_ids, mode):
    parts=[]
    for i, d in enumerate(doc_ids, 1):
        snippet = extractive_compress(question, d) if mode=='extractive' else abstractive_compress(question, d)
        if snippet:
            parts.append(f'[Document {i}] {snippet}')
    return '\n\n'.join(parts) if parts else '(no relevant snippets)'

def answer(question, context):
    sys = 'Answer only from the context. If not supported, reply NOT_FOUND. Keep short.'
    usr = f'Question: {question}\n\nContext:\n{context}\n\nReturn only the answer.'
    r = client.chat.completions.create(model=chat_model, temperature=0,
        messages=[{'role':'system','content':sys},{'role':'user','content':usr}])
    return r.choices[0].message.content.strip()

## 5. 单条样本直观感受

In [ ]:
sample = eval_rows[0]
dids = hybrid(sample['question'], top_k=5)
raw = build_raw_context(dids)
ext = build_compressed_context(sample['question'], dids, 'extractive')
absc = build_compressed_context(sample['question'], dids, 'abstractive')

print('question:', sample['question'])
print('reference:', sample['reference_answer'])
print('\n--- raw context (chars):', len(raw), '---')
print(raw[:400], '...')
print('\n--- extractive (chars):', len(ext), '---')
print(ext)
print('\n--- abstractive (chars):', len(absc), '---')
print(absc)

print('\nanswer raw        :', answer(sample['question'], raw))
print('answer extractive :', answer(sample['question'], ext))
print('answer abstractive:', answer(sample['question'], absc))

## 6. 小规模评估：raw vs extractive vs abstractive

重点看三件事：
- 答案质量（EM / F1）有没有掉
- context 长度有没有显著缩短
- 哪种压缩更稳

In [ ]:
def norm(t):
    return ' '.join([re.sub(r'[^a-z0-9]+','',x.lower()) for x in t.split() if re.sub(r'[^a-z0-9]+','',x.lower())])
def em(p,r): return 1.0 if norm(p)==norm(r) else 0.0
def f1(p,r):
    pt=norm(p).split(); rt=norm(r).split()
    if not pt or not rt: return 0.0
    ov=sum((Counter(pt)&Counter(rt)).values())
    if ov==0: return 0.0
    pr,rc=ov/len(pt),ov/len(rt); return 2*pr*rc/(pr+rc)

eval_subset = eval_rows[:5]
rows=[]
for i, r in enumerate(eval_subset, 1):
    print(f'{i}/{len(eval_subset)}: {r["question"]}')
    dids = hybrid(r['question'], top_k=5)
    raw = build_raw_context(dids)
    ext = build_compressed_context(r['question'], dids, 'extractive')
    absc = build_compressed_context(r['question'], dids, 'abstractive')
    a_raw = answer(r['question'], raw)
    a_ext = answer(r['question'], ext)
    a_abs = answer(r['question'], absc)
    rows.append({'q':r['question'],'ref':r['reference_answer'],
                 'len':(len(raw),len(ext),len(absc)),
                 'ans':(a_raw,a_ext,a_abs),
                 'em':(em(a_raw,r['reference_answer']),em(a_ext,r['reference_answer']),em(a_abs,r['reference_answer'])),
                 'f1':(f1(a_raw,r['reference_answer']),f1(a_ext,r['reference_answer']),f1(a_abs,r['reference_answer']))})

labels=['raw','extractive','abstractive']
print('\n=== summary ===')
for i,lab in enumerate(labels):
    print(f'{lab:12s}  avg_chars={np.mean([r["len"][i] for r in rows]):.0f}  EM={np.mean([r["em"][i] for r in rows]):.3f}  F1={np.mean([r["f1"][i] for r in rows]):.3f}')

## 7. 工程直觉

1. **Extractive 最安全**：不改写意味着幻觉几乎不会被引入。
2. **Abstractive 更省 token**，但要靠 prompt 约束住它。
3. 压缩常见的副作用是**把关键数字、日期、专有名词抹掉**。所以生产里最好强制要求保留数字/实体。
4. 代价：每个 top-k 文档都要多调一次 LLM，延迟和成本都会上。可以和检索并行、或者只对 top-2 压缩来折中。

下一课：**Conversational RAG**，也就是处理多轮对话里那些"它是什么意思" / "那他呢" 这种指代问题。